In [0]:
from pyspark.sql import functions as F
from pyspark.sql import *

In [0]:
source_path = '/Volumes/fineguard/source/fraud_watchlist/source_data/'

In [0]:
dbutils.fs.ls('/Volumes/fineguard/source/fraud_watchlist/source_data/')

In [0]:
input_sream = (
            spark.readStream.format('cloudFiles')
            .option('cloudFiles.format', 'json')
            .option('cloudFiles.schemaLocation', '/Volumes/fineguard/source/fraud_watchlist/schema/')
            .option('cloudFiles.inferColumnTypes', True)
            .load(source_path)
)

In [0]:
transformed_df = (
    input_sream
    .withColumn("file_path", F.col("_metadata.file_path"))
    .withColumn("ingestion_time", F.current_timestamp())."*"
)


In [0]:
streaming_data = (
    transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "dbfs:/Volumes/fineguard/source/fraud_watchlist/checkpoint/")
    .trigger(availableNow=True)
    .toTable("fineguard.bronze.fraud_watchlist_batch_test")
)


In [0]:
%sql
select * from fineguard.bronze.fraud_watchlist


In [0]:
cols = spark.table("fineguard.bronze.fraud_watchlist").columns
print(cols)
# Output: ['id', 'reason_code', 'file_path', 'ingestion_time']


In [0]:
%sql
drop table  fineguard.silver.fraud_watchlist